# Chapter 14 — Human-in-the-Loop Interactions

Hands-On LangChain. Code targets **LangGraph 1.x**.

We've built a capable agent that uses tools to answer questions. But in many real-world scenarios we need
more control: to **approve** an agent's actions before it executes them, **correct** its course if it
misunderstands, or **explore** alternative paths. This is **Human in the Loop (HITL)**, and LangGraph is
built for it — providing tools to pause, inspect, and manipulate an agent's state.

This chapter covers four HITL patterns:

- **Manual approval** — interrupt execution before a critical step (like a tool call) for human review.
- **State modification** — directly edit the agent's state to correct or steer it.
- **Time travel** — rewind to a previous state and branch off to explore a new path.
- **Injecting tool outputs** — manually provide a tool result, bypassing the real tool call.

Companion notebook for Chapter 14. The checkpointer is what makes all of this possible.

## 14.1 Project setup and dependencies

Configure Ollama and pull the chat model set in `MODEL` in `.env` before running this notebook. For HITL
we often need to **modify or replace** messages in the state, so instead of the default `operator.add`
reducer (which only appends), we use a custom `reduce_messages` reducer: if an incoming message has the
same `id` as an existing one, it **replaces** it; otherwise it appends. This lets us overwrite previous
steps.

In [1]:
from dotenv import load_dotenv, find_dotenv
from uuid import uuid4
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from model_provider import get_chat_model
from langchain_community.tools import DuckDuckGoSearchResults

load_dotenv(find_dotenv(usecwd=True))

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # give any message without an id a unique one
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # replace messages with the same id, otherwise append
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

tool = DuckDuckGoSearchResults(num_results=2, output_format="list")

The agent is the same as before, with one crucial change at compile time: **`interrupt_before=["action"]`**
tells LangGraph to pause right before the `action` node runs. Since `action` executes tools, this
creates an **approval gate**.

In [2]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.add_edge(START, "llm")
        self.graph = graph.compile(checkpointer=checkpointer, interrupt_before=["action"])
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_model(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        return {"messages": [self.model.invoke(messages)]}

    def exists_action(self, state: AgentState):
        return len(state["messages"][-1].tool_calls) > 0

    def take_action(self, state: AgentState):
        results = []
        for t in state["messages"][-1].tool_calls:
            print(f"Calling: {t['name']} {t['args']}")
            result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("Back to the model!")
        return {"messages": results}

prompt = """You are a smart research assistant. Use the search engine to look up information. You are allowed to make multiple calls (either together or in sequence). Only look up information when you are sure of what you want. If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
memory = InMemorySaver()
model = get_chat_model(temperature=0)
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

A small helper to print streamed events readably (instead of the full raw state dicts):

In [3]:
def show(event):
    for node, v in event.items():
        if not isinstance(v, dict) or not v.get("messages"):
            continue          # skip non-message events like __interrupt__
        m = v["messages"][-1]
        if getattr(m, "tool_calls", None):
            tc = m.tool_calls[0]
            print(f"[AI wants tool] {tc['name']}({tc['args']})")
        elif m.__class__.__name__ == "ToolMessage":
            print(f"[Tool result] {str(m.content)[:80]} ...")
        else:
            print(f"[AI] {m.content[:200]}")

## 14.2 Interrupting execution for manual approval

Run a request. Because of `interrupt_before=["action"]`, the graph stops **before** executing the tool.

In [ ]:
messages = [HumanMessage(content="What's the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}

for event in abot.graph.stream({"messages": messages}, thread):
    show(event)

Execution stopped after the AI decided to call the tool. We can inspect the paused state — the `next` attribute tells us what it's waiting to run:

In [ ]:
snapshot = abot.graph.get_state(thread)
print("next:", snapshot.next)
print("pending tool call:", snapshot.values["messages"][-1].tool_calls)

`('action',)` confirms it's paused, waiting to run the action node. To **approve** and continue, we stream again with `None` as input and the same thread — LangGraph picks up where it left off.

In [ ]:
for event in abot.graph.stream(None, thread):
    show(event)

## 14.3 Modifying state on the fly

Suppose the agent misinterprets "LA" as *Los Angeles* when we meant *Louisiana*. We can intervene. Run
until the interrupt, then inspect the tool call it's about to make.

In [ ]:
messages = [HumanMessage(content="What's the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    show(event)

current = abot.graph.get_state(thread)
print("about to search:", current.values["messages"][-1].tool_calls[0]["args"])

Let's change the query to point to Louisiana, then commit it back to memory with `update_state` (this is where our `reduce_messages` reducer matters — it **replaces** the message with the same id rather than appending):

In [ ]:
_id = current.values["messages"][-1].tool_calls[0]["id"]
current.values["messages"][-1].tool_calls = [
    {"name": "duckduckgo_results_json",
     "args": {"query": "current weather in Louisiana"},
     "id": _id}
]
abot.graph.update_state(thread, current.values)

# resume — the agent now runs our corrected action
for event in abot.graph.stream(None, thread):
    show(event)

The agent now searches for Louisiana instead of Los Angeles — we corrected its course mid-flight.

## 14.4 Time travel: rewinding and branching

As the graph runs, the checkpointer saves a snapshot at every step. Each snapshot's `config` uniquely
identifies that point, so we can **rewind** to it and resume — even branch off a new path.

Let's start a fresh thread, run to the interrupt, and look at the state history.

In [ ]:
messages = [HumanMessage(content="What's the weather in LA?")]
thread = {"configurable": {"thread_id": "tt"}}
for event in abot.graph.stream({"messages": messages}, thread):
    show(event)

history = list(abot.graph.get_state_history(thread))
print(f"\n{len(history)} snapshots in history (newest first):")
for i, s in enumerate(history):
    print(f"  [{i}] next={s.next}  #messages={len(s.values.get('messages', []))}")

We pick the snapshot that is paused **before the action** (the one with a pending tool call) as our replay point. Resuming from its `config` re-runs the original plan.

In [ ]:
to_replay = next(s for s in history if s.next == ("action",))
print("replay point pending query:", to_replay.values["messages"][-1].tool_calls[0]["args"])

print("\n--- resuming from the past snapshot (original plan) ---")
for event in abot.graph.stream(None, to_replay.config):
    show(event)

### Go back in time and edit

We can combine time travel with state modification to create a **new branch**. Let's return to the same
`to_replay` point but change the query first — say, to ask AccuWeather specifically.

In [ ]:
_id = to_replay.values["messages"][-1].tool_calls[0]["id"]
to_replay.values["messages"][-1].tool_calls = [
    {"name": "duckduckgo_results_json",
     "args": {"query": "current weather in LA accuweather"},
     "id": _id}
]
# update_state from a PAST config creates a new branch from that point
branch = abot.graph.update_state(to_replay.config, to_replay.values)

for event in abot.graph.stream(None, branch):
    show(event)

## 14.5 Manually adding tool results

A powerful pattern: provide a tool's output **yourself**, bypassing the real call — useful for mocking,
saving cost on expensive tools, or supplying data the agent can't fetch. We go back to `to_replay` and,
instead of editing the query, add a `ToolMessage` pretending the tool already ran and returned
"54 degree celcius".

The key is **`as_node="action"`**: it tells the graph to treat this update as the output of the `action`
node, so it knows the next step is to return to the LLM.

In [ ]:
_id = to_replay.values["messages"][-1].tool_calls[0]["id"]
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="duckduckgo_results_json",
    content="54 degree celcius",
)]}

branch_and_add = abot.graph.update_state(to_replay.config, state_update, as_node="action")

for event in abot.graph.stream(None, branch_and_add):
    show(event)

The agent skipped the tool call entirely and answered using our injected data.

You now have a full HITL toolkit: **interrupt** for approval, **modify state** to correct course,
**time travel** to revisit and branch, and **inject** data to guide the agent. These patterns turn an
agent from a black box into an interactive, steerable collaborator.

So far we've worked with a single agent. In the final LangGraph chapter we apply everything to a more
complex challenge: a **multi-agent system**.